# Thực nghiệm và đánh giá FPClose

Notebook chạy 6 thực nghiệm của Chương 2, so FPClose của nhóm (`src/`) với SPMF FPClose:

1. Correctness: so từng itemset và support với SPMF
2. Thời gian chạy theo minsup
3. Số lượng itemset theo minsup
4. Bộ nhớ đỉnh: bản cơ bản và bản tối ưu
5. Scalability trên Retail subsets
6. Ảnh hưởng của độ dài giao dịch trung bình

Mỗi mục có: mục tiêu (theo đề), dữ liệu, cách đo, cell **TODO** để tự viết code, file CSV kết quả và ô **Nhận xét**.
Kết quả lưu vào `data/results/experiments/` để vẽ lại biểu đồ mà không phải chạy lại.

**Cần trước khi chạy**
- Dữ liệu: đã chạy `data/notebooks/data_setup.ipynb` (có `data/benchmark/`, `data/scalability/`, `data/synthetic/`).
- Java 25 và `tools/spmf.jar` (xem `data/notebooks/reference_check.ipynb`).
- Kernel Julia dùng project của repo.

## 0 · Thiết lập

Tìm thư mục repo, nạp FPClose của nhóm, đặt đường dẫn dữ liệu, công cụ và nơi lưu kết quả.

In [4]:
using Pkg, Printf

# Thư mục gốc repo: nơi có Project.toml và src/FPClose.jl
function find_repo(dir = pwd())
    while true
        isfile(joinpath(dir, "src", "FPClose.jl")) && return dir
        parent = dirname(dir)
        parent == dir && error("Không tìm thấy repo FPClose (src/FPClose.jl). Mở notebook từ trong repo.")
        dir = parent
    end
end
ROOT = find_repo()
Pkg.activate(ROOT; io = devnull)
using FPClose

BENCHMARK_DIR   = joinpath(ROOT, "data", "benchmark")
SCALABILITY_DIR = joinpath(ROOT, "data", "scalability")
SYNTHETIC_DIR   = joinpath(ROOT, "data", "synthetic")
CONFIG_CSV      = joinpath(ROOT, "data", "results", "benchmark_config.csv")
RESULTS_DIR     = mkpath(joinpath(ROOT, "data", "results", "experiments"))
TMP_DIR         = mkpath(joinpath(ROOT, "out", "experiments"))   # output tạm, không commit

SPMF_JAR = joinpath(ROOT, "tools", "spmf.jar")
"Tìm java của JDK 25 (Temurin); nếu không có thì dùng `java` trên PATH."
function find_java()
    for root in ("C:\\Program Files\\Eclipse Adoptium", "/usr/lib/jvm")
        isdir(root) || continue
        for d in sort(readdir(root); rev = true)
            (startswith(d, "jdk-25") || startswith(d, "temurin-25")) || continue
            for exe in ("java.exe", "java")
                p = joinpath(root, d, "bin", exe)
                isfile(p) && return p
            end
        end
    end
    return "java"
end
JAVA = find_java()

DENSE  = ["chess", "connect", "mushrooms", "pumsb_star", "accidents"]
SPARSE = ["retail", "T10I4D100K", "T40I10D100K", "kosarak"]

for (name, p) in ("spmf.jar" => SPMF_JAR, "benchmark/" => BENCHMARK_DIR, "scalability/" => SCALABILITY_DIR,
                  "synthetic/" => SYNTHETIC_DIR)
    println(rpad(name, 14), ispath(p) ? "có   " : "THIẾU", "  ", p)
end
println(rpad("java", 14), JAVA)

spmf.jar      có     d:\7st_semester\Data\FPClose\tools\spmf.jar
benchmark/    có     d:\7st_semester\Data\FPClose\data\benchmark
scalability/  có     d:\7st_semester\Data\FPClose\data\scalability
synthetic/    có     d:\7st_semester\Data\FPClose\data\synthetic
java          C:\Program Files\Eclipse Adoptium\jdk-25.0.5.10002-hotspot\bin\java.exe


## Hàm phụ

Các hàm dùng chung cho mọi thực nghiệm. Đọc kỹ docstring để biết mỗi hàm trả về gì.

In [5]:
"Đọc CSV đơn giản (không có dấu phẩy trong ô) thành danh sách Dict."
function read_csv_rows(path)
    lines = filter(!isempty, strip.(readlines(path)))
    header = split(lines[1], ',')
    return [Dict(String(h) => String(v) for (h, v) in zip(header, split(l, ','))) for l in lines[2:end]]
end

"Ghi CSV (LF). `rows` là danh sách NamedTuple cùng tên trường."
function write_results(path, rows)
    isempty(rows) && return path
    cols = collect(keys(first(rows)))
    open(path, "w") do io
        println(io, join(cols, ','))
        for r in rows
            println(io, join((coalesce(getfield(r, c), "") for c in cols), ','))
        end
    end
    println("Đã ghi ", path)
    return path
end

"56 cấu hình (dataset, minsup %, minsup_count, số closed tham chiếu) từ benchmark_config.csv."
function benchmark_configs()
    [(dataset = r["dataset"], minsup_pct = parse(Float64, r["minsup_pct"]),
      minsup_count = parse(Int, r["minsup_count"]),
      reference = parse(Int, r["reference_closed_itemsets"])) for r in read_csv_rows(CONFIG_CSV)]
end

"Chuỗi minsup kiểu SPMF: 90.0 -> \"90%\", 0.05 -> \"0.05%\"."
pct_arg(p) = (s = string(p); endswith(s, ".0") ? s[1:end-2] * "%" : s * "%")

"""
    run_spmf(input, minsup_pct; algorithm = "FPClose") -> NamedTuple

Chạy SPMF một lần và đọc thống kê SPMF tự in: `time_ms` (Total time, không gồm khởi động JVM),
`memory_mb` (Max memory usage của JVM), `count` (số itemset) và `output` (đường dẫn file kết quả).
`algorithm`: "FPClose" (closed) hoặc "FPGrowth_itemsets" (mọi frequent itemset).
"""
function run_spmf(input, minsup_pct; algorithm = "FPClose", timeout_s = 600)
    out = joinpath(TMP_DIR, "spmf_$(algorithm)_$(splitext(basename(input))[1])_$(minsup_pct).txt")
    log = IOBuffer()
    proc = run(pipeline(`$JAVA -Xmx4g -jar $SPMF_JAR run $algorithm $input $out $(pct_arg(minsup_pct))`;
                        stdout = log, stderr = log); wait = false)
    t0 = time()
    while process_running(proc)
        if time() - t0 > timeout_s
            kill(proc); wait(proc)
            return (status = "TIMEOUT", time_ms = missing, memory_mb = missing, count = missing, output = out)
        end
        sleep(0.05)
    end
    text = String(take!(log))
    success(proc) || return (status = "FAILED", time_ms = missing, memory_mb = missing, count = missing, output = out)
    grab(re) = (m = match(re, text); m === nothing ? missing : parse(Float64, m[1]))
    return (status = "OK", time_ms = grab(r"Total time ~\s*([\d.]+)\s*ms"),
            memory_mb = grab(r"Max memory usage:\s*([\d.]+)"),
            count = (m = match(r"(?:itemset|itemsets) count\s*:\s*(\d+)", text); m === nothing ? missing : parse(Int, m[1])),
            output = out)
end

"Đọc output `i1 i2 ... #SUP: s` thành Dict(itemset đã sắp => support); bỏ dòng `@...`."
function read_itemsets(path)
    d = Dict{Vector{Int},Int}()
    for line in eachline(path)
        s = strip(line)
        (isempty(s) || startswith(s, '@')) && continue
        a, b = split(s, "#SUP:")
        d[sort!(parse.(Int, split(a)))] = parse(Int, strip(b))
    end
    return d
end

"Kết quả của nhóm (`itemset => support`) thành Dict giống `read_itemsets`."
as_dict(results) = Dict{Vector{Int},Int}(sort(collect(p.first)) => p.second for p in results)

"""
    compare_itemsets(ours, spmf) -> NamedTuple

So hai Dict itemset => support: `same` (cùng itemset, cùng support), `only_ours`, `only_spmf`,
`diff_support`, và `match_pct` = same / số itemset khác nhau của cả hai phía × 100.
"""
function compare_itemsets(ours::Dict, spmf::Dict)
    both = intersect(keys(ours), keys(spmf))
    same = count(k -> ours[k] == spmf[k], both)
    total = length(union(keys(ours), keys(spmf)))
    return (same = same, only_ours = length(setdiff(keys(ours), keys(spmf))),
            only_spmf = length(setdiff(keys(spmf), keys(ours))),
            diff_support = length(both) - same, match_pct = total == 0 ? 100.0 : 100 * same / total)
end

"""
    run_team_benchmark(input, minsup_pcts; combos = "ablation", reps = 3) -> Vector{Dict}

Gọi `scripts/benchmark.jl` của nhóm: mỗi (minsup, tổ hợp tối ưu) chạy trong một tiến trình Julia riêng,
có warm-up JIT, lấy thời gian nhỏ nhất của `reps` lần. Trả về các dòng CSV: `time_s`, `alloc_mb`,
`peak_rss_mb`, `delta_rss_mb`, `n_cfis`, và các cột `prune_line8`, `fparray`, `early_termination`.
"""
function run_team_benchmark(input, minsup_pcts; combos = "ablation", reps = 3)
    name = splitext(basename(input))[1]
    csv = joinpath(TMP_DIR, "benchmark_$(name).csv")
    minsups = join(pct_arg.(minsup_pcts), ',')
    run(`$(Base.julia_cmd()) --project=$ROOT $(joinpath(ROOT, "scripts", "benchmark.jl")) --minsups $minsups --combos $combos --reps $reps --csv $csv $input`)
    return read_csv_rows(csv)
end

"Lọc các dòng của `run_team_benchmark` ứng với bản tối ưu đầy đủ (bật cả 3 công tắc)."
optimized(rows) = filter(r -> r["prune_line8"] == r["fparray"] == r["early_termination"] == "true", rows)

optimized (generic function with 1 method)

### Kiểm tra nhanh các hàm phụ

Chạy SPMF và FPClose của nhóm trên chess 90% rồi so. Nếu cell này lỗi thì sửa thiết lập trước khi làm tiếp.

In [6]:
s = run_spmf(joinpath(BENCHMARK_DIR, "chess.txt"), 90.0)
println(s)
ours = as_dict(fpclose(read_spmf(joinpath(BENCHMARK_DIR, "chess.txt")), 2877))
println(compare_itemsets(ours, read_itemsets(s.output)))

(status = "OK", time_ms = 65.0, memory_mb = 32.1036376953125, count = 498, output = "d:\\7st_semester\\Data\\FPClose\\out\\experiments\\spmf_FPClose_chess_90.0.txt")
(same = 498, only_ours = 0, only_spmf = 0, diff_support = 0, match_pct = 100.0)


## 1 · Correctness

**Mục tiêu (đề):** với mỗi tập dữ liệu, so số lượng itemset và support của từng itemset giữa cài đặt của nhóm và SPMF.
Báo cáo tỉ lệ itemset khớp hoàn toàn; nếu có sai lệch, phân tích nguyên nhân.

**Dữ liệu:** 9 benchmark × 56 cấu hình (`benchmark_configs()`).

**Cách đo**
- Closed itemsets: `fpclose(db, minsup_count)` của nhóm so với `run_spmf(...; algorithm = "FPClose")`.
- Mọi frequent itemset (đề viết "frequent itemset"): `recover_all_frequent(closed; limit = 5_000_000)` của nhóm so với
  `run_spmf(...; algorithm = "FPGrowth_itemsets")`. Trên dữ liệu dày ở minsup thấp số itemset bùng nổ: bắt lỗi
  `RecoveryLimitExceeded` và ghi lại cấu hình bị bỏ qua.
- Dùng `compare_itemsets` để lấy `match_pct`, `only_ours`, `only_spmf`, `diff_support`.

**Kết quả:** `data/results/experiments/exp1_correctness.csv` (một dòng mỗi cấu hình). Trong báo cáo: bảng % khớp theo dataset.

In [22]:
# TODO 1: với mỗi cấu hình c in benchmark_configs():
#   - db = read_spmf(joinpath(BENCHMARK_DIR, c.dataset * ".txt"))   (đọc 1 lần mỗi dataset)
#   - ours = as_dict(fpclose(db, c.minsup_count))
#   - s = run_spmf(input, c.minsup_pct); cmp = compare_itemsets(ours, read_itemsets(s.output))
#   - push!(rows, (dataset = ..., minsup_pct = ..., minsup_count = ..., n_ours = ..., n_spmf = ..., cmp...))
# rows = NamedTuple[]
# write_results(joinpath(RESULTS_DIR, "exp1_correctness.csv"), rows)
configs = benchmark_configs()
length(configs), configs[1]

(56, (dataset = "chess", minsup_pct = 90.0, minsup_count = 2877, reference = 498))

### Biểu đồ

In [8]:
# TODO: đọc CSV trong RESULTS_DIR và vẽ (thư viện vẽ: thống nhất với nhóm, ví dụ Plots.jl)

### Nhận xét

_TODO: số liệu chính, giải thích dựa trên bài báo và độ phức tạp, so sánh với SPMF._

## 2 · Thời gian chạy theo minsup

**Mục tiêu (đề):** với mỗi tập dữ liệu, vẽ thời gian chạy theo minsup giảm dần (5–7 điểm, từ cao xuống thấp),
đường của nhóm và của SPMF trên cùng đồ thị.

**Dữ liệu:** 9 benchmark, các mức minsup trong `benchmark_configs()`.

**Cách đo**
- Nhóm: `run_team_benchmark(input, minsups)`, lấy `time_s` của bản tối ưu (`optimized(rows)`).
- SPMF: `run_spmf(input, minsup).time_ms / 1000`. Đây là thời gian SPMF tự đo, không gồm khởi động JVM, nhưng
  **có gồm ghi file output**; `fpclose` của nhóm thì không ghi file. Ghi rõ điều này trong báo cáo.
- Nên chạy SPMF vài lần và lấy nhỏ nhất, giống phía nhóm.

**Kết quả:** `exp2_time.csv` (dataset, minsup_pct, team_time_s, spmf_time_s). Biểu đồ: 9 đồ thị, trục y log.

In [9]:
# TODO 2: với mỗi dataset, minsups = [c.minsup_pct for c in benchmark_configs() if c.dataset == d]
#   - team = optimized(run_team_benchmark(input, minsups))
#   - spmf = [minimum(run_spmf(input, m).time_ms for _ in 1:3) / 1000 for m in minsups]
# write_results(joinpath(RESULTS_DIR, "exp2_time.csv"), rows)

### Biểu đồ

In [10]:
# TODO: đọc CSV trong RESULTS_DIR và vẽ (thư viện vẽ: thống nhất với nhóm, ví dụ Plots.jl)

### Nhận xét

_TODO: số liệu chính, giải thích dựa trên bài báo và độ phức tạp, so sánh với SPMF._

## 3 · Số lượng itemset theo minsup

**Mục tiêu (đề):** vẽ số lượng itemset sinh ra theo minsup; nhận xét quan hệ giữa minsup và output size,
so sánh dữ liệu dày (`DENSE`) và thưa (`SPARSE`).

**Dữ liệu:** số closed itemsets đã có trong `benchmark_configs()` (`reference`, đã được kiểm ở thực nghiệm 1).
Nếu muốn thêm đường "mọi frequent itemset", lấy từ thực nghiệm 1.

**Kết quả:** `exp3_counts.csv`. Biểu đồ: hai đồ thị (dày, thưa), trục y log; có thể thêm tỉ lệ closed / frequent.

In [11]:
# TODO 3: rows = [(dataset = c.dataset, minsup_pct = c.minsup_pct, n_closed = c.reference,
#                  density = c.dataset in DENSE ? "dense" : "sparse") for c in benchmark_configs()]
# write_results(joinpath(RESULTS_DIR, "exp3_counts.csv"), rows)

### Biểu đồ

In [12]:
# TODO: đọc CSV trong RESULTS_DIR và vẽ (thư viện vẽ: thống nhất với nhóm, ví dụ Plots.jl)

### Nhận xét

_TODO: số liệu chính, giải thích dựa trên bài báo và độ phức tạp, so sánh với SPMF._

## 4 · Bộ nhớ đỉnh

**Mục tiêu (đề):** đo và báo cáo RAM tối đa (peak memory) của mỗi tập dữ liệu tại minsup trung bình;
so sánh bản cơ bản và bản tối ưu của nhóm.

**Dữ liệu:** mỗi dataset một mức minsup ở giữa danh sách của nó.

**Cách đo**
- `run_team_benchmark(input, [m]; combos = "ablation")` cho 4 tổ hợp: cơ bản, +FP-array, +early termination, cả hai.
- Các cột: `peak_rss_mb` (cả tiến trình, khoảng 350 MB là Julia runtime), `delta_rss_mb` (phần tăng thêm khi khai thác),
  `alloc_mb` (tổng bộ nhớ cấp phát). Chọn chỉ số nào để vẽ và **giải thích lý do** trong báo cáo.
- Tham khảo thêm: `run_spmf(...).memory_mb` (heap JVM, đo theo cách khác).

**Kết quả:** `exp4_memory.csv`. Biểu đồ: cột ghép theo dataset × tổ hợp.

In [13]:
# TODO 4: với mỗi dataset: ms = các minsup của dataset; m = ms[cld(length(ms), 2)]
#   - rows_d = run_team_benchmark(input, [m]; combos = "ablation")
#   - giữ các cột prune_line8, fparray, early_termination, time_s, alloc_mb, peak_rss_mb, delta_rss_mb
# write_results(joinpath(RESULTS_DIR, "exp4_memory.csv"), rows)

### Biểu đồ

In [14]:
# TODO: đọc CSV trong RESULTS_DIR và vẽ (thư viện vẽ: thống nhất với nhóm, ví dụ Plots.jl)

### Nhận xét

_TODO: số liệu chính, giải thích dựa trên bài báo và độ phức tạp, so sánh với SPMF._

## 5 · Scalability

**Mục tiêu (đề):** trên một tập lớn (Retail), dùng các tập con 10%, 25%, 50%, 75%, 100% giao dịch;
vẽ thời gian chạy theo kích thước CSDL; nhận xét xu hướng tuyến tính hay phi tuyến.

**Dữ liệu:** `data/scalability/retail_p010.txt` … `retail_p100.txt`.

**Cách đo:** một minsup cố định (tỉ lệ %) cho cả 5 tập. **Cần chọn**: chạy thử vài mức trên `retail_p100`
để thời gian đủ lớn mà vẫn chạy xong. Đo nhóm (`run_team_benchmark`) và SPMF (`run_spmf`).

**Kết quả:** `exp5_scalability.csv` (size_pct, n_transactions, team_time_s, spmf_time_s). Biểu đồ: thời gian theo % kích thước.

In [15]:
# TODO 5: MINSUP_SCALE = ...   # chọn sau khi chạy thử
# for p in ("p010", "p025", "p050", "p075", "p100")
#     input = joinpath(SCALABILITY_DIR, "retail_$p.txt")
#     ...
# end
# write_results(joinpath(RESULTS_DIR, "exp5_scalability.csv"), rows)

### Biểu đồ

In [16]:
# TODO: đọc CSV trong RESULTS_DIR và vẽ (thư viện vẽ: thống nhất với nhóm, ví dụ Plots.jl)

### Nhận xét

_TODO: số liệu chính, giải thích dựa trên bài báo và độ phức tạp, so sánh với SPMF._

## 6 · Ảnh hưởng của độ dài giao dịch trung bình

**Mục tiêu (đề):** chứng minh (hoặc bác bỏ) việc thuật toán bị ảnh hưởng bởi độ dài giao dịch,
bằng CSDL tổng hợp có độ dài giao dịch tăng dần.

**Dữ liệu:** `data/synthetic/len05.txt` … `len30.txt` (10.000 giao dịch, 500 item, độ dài trung bình 5–30,
các mức lồng nhau: giao dịch dài chứa giao dịch ngắn). Thống kê trong `data/synthetic/synthetic_statistics.csv`.

**Cách đo:** một minsup cố định cho cả 6 tập (**cần chọn** bằng cách chạy thử, giống thực nghiệm 5).
Ghi thời gian của nhóm và SPMF, cùng số closed itemsets.

**Kết quả:** `exp6_length.csv` (avg_length, n_closed, team_time_s, spmf_time_s). Biểu đồ: thời gian và số itemset theo độ dài.

In [17]:
# TODO 6: MINSUP_LEN = ...   # chọn sau khi chạy thử
# for l in ("len05", "len10", "len15", "len20", "len25", "len30")
#     input = joinpath(SYNTHETIC_DIR, "$l.txt")
#     ...
# end
# write_results(joinpath(RESULTS_DIR, "exp6_length.csv"), rows)

### Biểu đồ

In [18]:
# TODO: đọc CSV trong RESULTS_DIR và vẽ (thư viện vẽ: thống nhất với nhóm, ví dụ Plots.jl)

### Nhận xét

_TODO: số liệu chính, giải thích dựa trên bài báo và độ phức tạp, so sánh với SPMF._

## 7 · Đánh giá chung và hướng tối ưu

_TODO_
- Điểm mạnh và điểm yếu của cài đặt so với SPMF, theo từng loại dữ liệu (dày, thưa, lớn).
- Ít nhất 2 hướng tối ưu cụ thể có thể làm tiếp (tham khảo `docs/DEVIATIONS.md`).